<a href="https://colab.research.google.com/github/taehi5/ESAA_YB_1/blob/Project3/et_gbr_ensemble(3_7).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import warnings
import numpy as np, pandas as pd
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold
warnings.filterwarnings("ignore")

tr = pd.read_csv("movies_train.csv")
te = pd.read_csv("movies_test.csv")
sub = pd.read_csv("submission.csv")

def clean_dist(s):
    return s.str.replace(r"\(주\)|㈜|\s", "", regex=True)

def feats(df, ref):
    X = pd.DataFrame(index=df.index)
    d = pd.to_datetime(df.release_time)
    X["year"] = d.dt.year
    X["month"] = d.dt.month.astype(str)
    X["time"] = df.time
    X["log_staff"] = np.log1p(df.num_staff)
    X["num_actor"] = df.num_actor
    X["debut"] = (df.dir_prev_num == 0).astype(int)
    X["log_prev_bf"] = np.log1p(df.dir_prev_bfnum.fillna(0))
    X["dir_prev_num"] = df.dir_prev_num
    top = clean_dist(ref.distributor).value_counts()
    keep = top[top >= 10].index
    dist = clean_dist(df.distributor)
    X["dist"] = np.where(dist.isin(keep), dist, "etc")
    X["genre"] = df.genre
    X["rat"] = df.screening_rat
    return X

cat = ["month", "dist", "genre", "rat"]
Xtr, Xte = feats(tr, tr), feats(te, tr)
y = tr.box_off_num.values

In [ ]:
def enc():
    return ColumnTransformer(
        [("c", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat)],
        remainder="passthrough")

def et_model():
    return make_pipeline(enc(), ExtraTreesRegressor(n_estimators=500, min_samples_leaf=2,
                                                    random_state=0, n_jobs=-1))

def gbr_model():
    return make_pipeline(enc(), GradientBoostingRegressor(loss="huber", n_estimators=300,
                                                          learning_rate=0.03, max_depth=3,
                                                          subsample=0.8, random_state=0))

In [ ]:
W_ET = 0.3  # ET 가중치

In [ ]:
def fit_predict(Xa, ya, Xb):
    p_et = np.clip(et_model().fit(Xa, ya).predict(Xb), 0, None)                          # 원본 타깃
    p_gbr = np.square(np.clip(gbr_model().fit(Xa, np.sqrt(ya)).predict(Xb), 0, None))   # √ 타깃
    return W_ET * p_et + (1 - W_ET) * p_gbr

In [ ]:
# CV (OOF RMSE)
for seed in [0, 1, 2]:
    oof = np.zeros(len(y))
    for a, b in KFold(5, shuffle=True, random_state=seed).split(Xtr):
        oof[b] = fit_predict(Xtr.iloc[a], y[a], Xtr.iloc[b])
    print(seed, f"{np.sqrt(np.mean((oof - y) ** 2)):,.0f}")

0 1,477,567
1 1,475,183
2 1,473,435


In [ ]:
# 제출
sub["box_off_num"] = fit_predict(Xtr, y, Xte)
sub.to_csv("submission_et_gbr(3:7).csv", index=False)